# 🧠 Pelatihan Machine Learning AI Toko Laptop (Intent Classification)

Notebook ini memandu Anda langkah demi langkah dalam melatih model **Machine Learning (NLP)** untuk memahami maksud (intent) pembeli di toko laptop:
1. **Mempersiapkan Dataset Percakapan** (Supervised Learning)
2. **Feature Extraction (TF-IDF Vectorization)**: Mengubah kata/teks menjadi representasi angka matematis
3. **Model Training**: Melatih algoritma Naive Bayes / Logistic Regression
4. **Evaluasi Akurasi**: Menguji model dengan data uji (Test Set) dan matriks klasifikasi
5. **Export Model (.pkl)**: Menyimpan model hasil pelatihan agar bisa langsung digunakan di aplikasi utama (`app.py`)
6. **Live Inference**: Mengetes model dengan kalimat bebas baru

In [ ]:
# 1. Import Pustaka Machine Learning Standar
import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## 2. Memuat Dataset Percakapan

In [ ]:
with open('dataset_chat_latihan.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Total data: {len(df)} sampel")
df.head(10)

## 3. Split Dataset (Train vs Test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['intent'],
    test_size=0.2,
    random_state=42,
    stratify=df['intent']
)

print(f"Data Latih (Train): {len(X_train)} baris")
print(f"Data Uji (Test)  : {len(X_test)} baris")

## 4. Ekstraksi Fitur Teks dengan TF-IDF

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Ukuran matriks fitur:", X_train_vec.shape)

## 5. Pelatihan Model (Training)

In [ ]:
model = MultinomialNB(alpha=0.3)
model.fit(X_train_vec, y_train)
print("✅ Model berhasil dilatih!")

## 6. Evaluasi Akurasi & Confusion Matrix

In [ ]:
y_pred = model.predict(X_test_vec)
acc = accuracy_score(y_test, y_pred)
print(f"Akurasi Model: {acc * 100:.2f}%")
print("\nLaporan Klasifikasi Detail:")
print(classification_report(y_test, y_pred, zero_division=0))

## 7. Simpan Model Hasil Latihan (.pkl)

In [ ]:
model_data = {
    'model': model,
    'vectorizer': vectorizer
}
joblib.dump(model_data, 'model_intent_laptop.pkl')
print("💾 Model berhasil disimpan ke 'model_intent_laptop.pkl'!")

## 8. Coba Prediksi Kalimat Baru Secara Langsung

In [ ]:
def tebak_maksud_chat(kalimat):
    vec = vectorizer.transform([kalimat])
    prediksi = model.predict(vec)[0]
    confidence = max(model.predict_proba(vec)[0]) * 100
    return prediksi, confidence

# Masukkan kalimat bebas di sini:
contoh_chat = "bro ada laptop 8 jutaan yang kuat buat main dota?"
intent, conf = tebak_maksud_chat(contoh_chat)
print(f"Kalimat: '{contoh_chat}'")
print(f"Hasil Prediksi Intent: {intent} (Keyakinan: {conf:.1f}%)")